# EFSM Quick Evaluation: Base vs Fine-Tuned

This notebook verifies that the 1-epoch LoRA adapter loads correctly and compares base Qwen2.5-7B-Instruct responses against the fine-tuned EFSM adapter on emotionally varied prompts.

## Cell 1 — Secrets

Requires Kaggle secret `HF_TOKEN` with read access to `tasbid001/efsm-checkpoints-fixed`.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
print('HF token loaded.')

## Cell 2 — Clone Latest Repo and Install

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/tasbidrahman10/empathetic-voice-llm.git'
REPO_DIR = '/kaggle/working/efsm-code-fixed'

os.chdir('/kaggle/working')
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
print('Repo commit:')
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Requirements installed.')
print('Torch CUDA available:', __import__('torch').cuda.is_available())

## Cell 3 — Run Comparison

This generates base responses first, unloads the model, then generates fine-tuned responses. Start with all 12 prompts; reduce `--limit` if you need a faster smoke test.

In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
# For inference, allow Transformers device_map='auto' to use both Kaggle T4 GPUs.
env.pop('CUDA_VISIBLE_DEVICES', None)
alloc_conf = 'expandable_segments:True,max_split_size_mb:64,garbage_collection_threshold:0.8'
env['PYTORCH_ALLOC_CONF'] = alloc_conf
env['PYTORCH_CUDA_ALLOC_CONF'] = alloc_conf

cmd = [
    sys.executable,
    'src/eval/quick_compare.py',
    '--config', 'configs/config.yaml',
    '--output', 'results/quick_eval_results.csv',
    '--limit', '12',
    '--max-new-tokens', '80',
    '--device-strategy', 'cpu_offload',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env)

## Cell 4 — Display Comparison Table

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 500)
df = pd.read_csv('results/quick_eval_results.csv')
display(df[['emotion', 'prompt', 'base_response', 'fine_tuned_response']])

## Cell 5 — Manual Scoring Rubric

For each row, score base and fine-tuned responses from 1 to 5:

- Emotional acknowledgement: does it name/validate the feeling?
- Warmth: does it sound caring and human?
- Relevance: does it respond to the specific situation?
- Therapeutic tone: does it avoid judgment and avoid rushing into advice?

For tomorrow's supervisor demo, show 5-8 strongest rows plus the W&B training curve.